## Extract

In [3]:
from pymongo import MongoClient
client = MongoClient("mongodb://localhost:27017/")

In [6]:
db = client["school"]
grades = list(db["grades"].find({}, {"_id": 0}))
attendances = list(db["attendances"].find({}, {"_id": 0}))
students = list(db["students"].find({}, {"_id": 0}))

In [8]:
print("Grades:", len(grades))
print("Attendances:", len(attendances))
print("Students:", len(students))

Grades: 105
Attendances: 104
Students: 100


In [11]:
client.close()

## Transform

In [12]:
import pandas as pd

In [23]:
grades_df = pd.DataFrame(grades)

for subject in ["math", "python"]:
    grades_df[subject] = pd.to_numeric(grades_df[subject], errors="coerce")
    grades_df.loc[~grades_df[subject].between(0, 100),subject] = float("nan")

grades_clean = grades_df.melt(
    id_vars=["student_id"],
    value_vars=["math", "python"],
    var_name="subject",
    value_name="grade"
)
grades_clean = grades_clean.dropna(subset=["grade"])
grades_clean["grade"] = grades_clean["grade"].astype(int)
print("\nClean Grades:")
print(grades_clean.head(10))
print("Valid grade records:", len(grades_clean))
print("\nMissing values per column:")
print(grades_clean.isna().sum())


Clean Grades:
   student_id subject  grade
0           1    math     90
1           2    math     80
2           3    math     84
3           4    math     75
4           5    math     70
5           6    math     70
6           7    math     95
7           8    math     60
8           9    math     83
9          10    math     78
Valid grade records: 207

Missing values per column:
student_id    0
subject       0
grade         0
dtype: int64


In [31]:
attendance_df = pd.DataFrame(attendances)

attendance_df["status"] = (attendance_df["status"].fillna("").astype(str).str.strip().str.title())

attendance_df.loc[~attendance_df["status"].isin(["Present", "Absent"]),"status"] = "Unknown"

attendance_df["date"] = pd.to_datetime(attendance_df["date"], errors="coerce").dt.date

attendance_clean = attendance_df.dropna(subset=["date"]).copy()

print("\nClean Attendance:")
print(attendance_clean.head(10))
print("Attendance records:", len(attendance_clean))
print("\nMissing values per column:")
print(attendance_df.isna().sum())



Clean Attendance:
   student_id        date   status note
0           1  2026-10-01  Present  NaN
1           2  2026-10-01   Absent  NaN
2           3  2026-10-01  Present  NaN
3           4  2026-10-01  Present  NaN
4           5  2026-10-01   Absent  NaN
5           6  2026-10-01  Present  NaN
6           7  2026-10-01  Present  NaN
7           8  2026-10-01   Absent  NaN
8           9  2026-10-01  Present  NaN
9          10  2026-10-01  Present  NaN
Attendance records: 104

Missing values per column:
student_id     0
date           0
status         0
note          88
dtype: int64


In [25]:
students_df = pd.DataFrame(students)

students_df.columns = students_df.columns.str.strip().str.lower()
students_df["age"] = pd.to_numeric(students_df["age"], errors="coerce")
students_df["gpa"] = pd.to_numeric(students_df["gpa"], errors="coerce")

students_df.loc[~students_df["age"].between(1, 100),"age"] = float("nan")

students_df.loc[ ~students_df["gpa"].between(0, 4),"gpa"] = float("nan")

for column in ["name", "city", "gender", "level"]:
    if column in students_df.columns:
        students_df[column] = (students_df[column].astype("string").str.strip().replace("", pd.NA))

print("\nClean Students:")
print(students_df.head(10))

print("\nMissing values per column:")
print(students_df.isna().sum())



Clean Students:
   student_id     name   age        city  gpa email phone gender level
0           1  Mohamed  22.0       Cairo  3.5   NaN   NaN   <NA>  <NA>
1           2    Ahmed  23.0        Giza  3.2   NaN   NaN   <NA>  <NA>
2           3      Mai  21.0  Alexandria  3.8   NaN   NaN   <NA>  <NA>
3           4     Sara  24.0       Cairo  3.6   NaN   NaN   <NA>  <NA>
4           5     Omar  22.0       Luxor  3.1   NaN   NaN   <NA>  <NA>
5           6  Youssef  20.0       Aswan  3.4   NaN   NaN   <NA>  <NA>
6           7     Hana  19.0       Cairo  3.9   NaN   NaN   <NA>  <NA>
7           8    Tarek  25.0        Giza  3.0   NaN   NaN   <NA>  <NA>
8           9     Nour  22.0  Alexandria  3.7   NaN   NaN   <NA>  <NA>
9          10    Laila  23.0       Cairo  3.5   NaN   NaN   <NA>  <NA>

Missing values per column:
student_id     0
name           0
age            6
city           0
gpa            0
email         62
phone         62
gender        62
level         62
dtype: int64


In [15]:
if students:
    student_ids = {
        student["student_id"]
        for student in students
        if "student_id" in student
    }

    orphan_grades = grades_clean[~grades_clean["student_id"].isin(student_ids)]
    orphan_attendance = attendance_clean[~attendance_clean["student_id"].isin(student_ids)]

    print("\nOrphan grade records:", len(orphan_grades))
    print("Orphan attendance records:", len(orphan_attendance))
else:
    print( "\nStudent reference check skipped: "
        "the students collection is empty."
    )

print("\nAttendance status counts:")
print(attendance_clean["status"].value_counts())

print("\nAverage grade by subject:")
print(
    grades_clean.groupby("subject")["grade"]
    .mean()
    .round(2)
)



Orphan grade records: 10
Orphan attendance records: 4

Attendance status counts:
status
Present    69
Absent     33
Unknown     2
Name: count, dtype: int64

Average grade by subject:
subject
math      77.82
python    81.56
Name: grade, dtype: float64


In [16]:
print("\nAttendance status counts:")
print(attendance_clean["status"].value_counts())

print("\nAverage grade by subject:")
print(grades_clean.groupby("subject")["grade"].mean().round(2))


Attendance status counts:
status
Present    69
Absent     33
Unknown     2
Name: count, dtype: int64

Average grade by subject:
subject
math      77.82
python    81.56
Name: grade, dtype: float64


In [ ]:
attendance_df = attendance_df.drop_duplicates(subset=["student_id", "date"])
students_df = students_df.drop_duplicates(subset=["student_id"],keep="first")

## Load

In [39]:
excluded_grades = grades_clean[ ~grades_clean["student_id"].isin(valid_student_ids)]

excluded_attendance = attendance_clean[~attendance_clean["student_id"].isin(valid_student_ids)]

print("Excluded grade records:")
print(excluded_grades)

print("\nExcluded attendance records:")
print(excluded_attendance)

Excluded grade records:
     student_id subject  grade
100         101    math     60
101         102    math     71
102         103    math     70
103         104    math     65
104         999    math     85
205         101  python     65
206         102  python     77
207         103  python     75
208         104  python     72
209         999  python     80

Excluded attendance records:
     student_id        date   status    note
100         101  2026-10-13  Present     NaN
101         102  2026-10-13  Unknown     NaN
102         103  2026-10-13  Unknown     NaN
103         999  2026-10-13  Present  orphan


In [36]:
import pyodbc
conn = pyodbc.connect(
    "DRIVER={ODBC Driver 18 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=StudentPerformance;"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

cursor = conn.cursor()

In [37]:
import pandas as pd

valid_student_ids = set(students_df["student_id"])

grades_to_load = grades_clean[grades_clean["student_id"].isin(valid_student_ids)].copy()

attendance_to_load = attendance_clean[attendance_clean["student_id"].isin(valid_student_ids)].copy()

def sql_value(value):
    if pd.isna(value):
        return None
    if hasattr(value, "item"):
        return value.item()
    return value


student_rows = [
    tuple(sql_value(v) for v in row)
    for row in students_df[["student_id", "name", "age", "city", "gpa","email", "phone", "gender", "level"]].itertuples(index=False, name=None)
]

cursor.executemany(
    """
    INSERT INTO dbo.Students
    (student_id, name, age, city, gpa,
     email, phone, gender, level)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
    """,
    student_rows
)


grade_rows = [
    tuple(sql_value(v) for v in row)
    for row in grades_to_load[["student_id", "subject", "grade"]].itertuples(index=False, name=None)
]

cursor.executemany(
    """
    INSERT INTO dbo.Grades (student_id, subject, grade)
    VALUES (?, ?, ?)
    """,
    grade_rows
)


attendance_rows = [
    tuple(sql_value(v) for v in row)
    for row in attendance_to_load[["student_id", "date", "status", "note"]].itertuples(index=False, name=None)]

cursor.executemany(
    """
    INSERT INTO dbo.Attendance (student_id, date, status, note)
    VALUES (?, ?, ?, ?)
    """,
    attendance_rows
)

conn.commit()
print("All data loaded successfully")

All data loaded successfully


In [41]:
students_df.to_csv("Students.csv", index=False)
grades_to_load.to_csv("Grades.csv", index=False)
attendance_to_load.to_csv("Attendance.csv", index=False)